# LAB 3 — File Generation

This notebook imports the shared Lab 3 configuration, reads the NYC Yellow Taxi Parquet file from the external volume, and generates approximately 1,000 JSON files for Auto Loader testing.

## 1. Load shared configuration

In [0]:
%run ./lab03_config

## 2. Verify the uploaded source file

In [0]:
source_files = dbutils.fs.ls(source_path)
display(source_files)

matching_files = [
    file_info.path
    for file_info in source_files
    if file_info.name == source_file_name
]

if not matching_files:
    raise FileNotFoundError(
        f"Expected source file was not found: {source_file_path}"
    )

print("Source file found.")

In [0]:
%sql
SELECT *
FROM read_files(
  '/Volumes/dbr_dev/parvinbadalov/lab03_streaming/source/_READY',
  format => 'text'
)
LIMIT 20;

In [0]:
display(dbutils.fs.ls('dbfs:/Volumes/dbr_dev/parvinbadalov/lab03_streaming/source/'))

## 3. Read and inspect the Parquet dataset

In [0]:
taxi_df = spark.read.parquet(source_file_path)

display(taxi_df.limit(10))

In [0]:
taxi_df.printSchema()

total_rows = taxi_df.count()

print(f"Total rows: {total_rows:,}")
print(f"Total columns: {len(taxi_df.columns)}")

## 4. Select columns and limit the working dataset

In [0]:
from pyspark.sql.functions import col

required_columns = [
    "tpep_pickup_datetime",
    "tpep_dropoff_datetime",
    "passenger_count",
    "trip_distance",
    "PULocationID",
    "DOLocationID",
    "payment_type",
    "fare_amount",
    "tip_amount",
    "total_amount",
]

missing_columns = [
    column_name
    for column_name in required_columns
    if column_name not in taxi_df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

base_df = (
    taxi_df
    .select(
        col("tpep_pickup_datetime").alias("pickup_datetime"),
        col("tpep_dropoff_datetime").alias("dropoff_datetime"),
        col("passenger_count"),
        col("trip_distance"),
        col("PULocationID").alias("pickup_location_id"),
        col("DOLocationID").alias("dropoff_location_id"),
        col("payment_type"),
        col("fare_amount"),
        col("tip_amount"),
        col("total_amount"),
    )
    .limit(100_000)
)

display(base_df.limit(10))

## 5. Check null values

In [0]:
from pyspark.sql.functions import sum as spark_sum, when

null_counts_df = base_df.select(
    [
        spark_sum(
            when(col(column_name).isNull(), 1).otherwise(0)
        ).alias(column_name)
        for column_name in base_df.columns
    ]
)

display(null_counts_df)

## 6. Split the dataset into schema-test groups

In [0]:
initial_df, evolved_base_df, renamed_base_df, malformed_base_df = (
    base_df.randomSplit(
        [0.80, 0.10, 0.05, 0.05],
        seed=42
    )
)

split_counts = {
    "initial_rows": initial_df.count(),
    "evolved_rows": evolved_base_df.count(),
    "renamed_rows": renamed_base_df.count(),
    "malformed_rows": malformed_base_df.count(),
}

for name, value in split_counts.items():
    print(f"{name}: {value:,}")

## 7. Generate initial-schema files

In [0]:
(
    initial_df
    .repartition(800)
    .write
    .mode("overwrite")
    .json(staging_initial_path)
)

print("Initial-schema files generated.")

## 8. Generate evolved-schema files

In [0]:
from pyspark.sql.functions import lit

evolved_df = (
    evolved_base_df
    .withColumn("source_system", lit("nyc_tlc"))
)

(
    evolved_df
    .repartition(100)
    .write
    .mode("overwrite")
    .json(staging_evolved_path)
)

print("Evolved-schema files generated.")

## 9. Generate renamed-column files

In [0]:
renamed_df = (
    renamed_base_df
    .withColumnRenamed("fare_amount", "base_fare_amount")
)

(
    renamed_df
    .repartition(50)
    .write
    .mode("overwrite")
    .json(staging_renamed_path)
)

print("Renamed-column files generated.")

## 10. Generate malformed files

In [0]:
from pyspark.sql.functions import concat

malformed_df = (
    malformed_base_df
    .withColumn(
        "passenger_count",
        concat(
            lit("invalid_"),
            col("passenger_count").cast("string")
        )
    )
    .withColumn("unexpected_field", lit("schema_test"))
)

(
    malformed_df
    .repartition(50)
    .write
    .mode("overwrite")
    .json(staging_malformed_path)
)

print("Malformed files generated.")

## 11. Validate generated file counts

In [0]:
def count_json_files(path: str) -> int:
    return len(
        [
            file_info
            for file_info in dbutils.fs.ls(path)
            if file_info.name.endswith(".json")
        ]
    )

initial_file_count = count_json_files(staging_initial_path)
evolved_file_count = count_json_files(staging_evolved_path)
renamed_file_count = count_json_files(staging_renamed_path)
malformed_file_count = count_json_files(staging_malformed_path)

total_file_count = (
    initial_file_count
    + evolved_file_count
    + renamed_file_count
    + malformed_file_count
)

file_counts_df = spark.createDataFrame(
    [
        ("initial", initial_file_count),
        ("evolved", evolved_file_count),
        ("renamed", renamed_file_count),
        ("malformed", malformed_file_count),
        ("total", total_file_count),
    ],
    ["file_group", "json_file_count"]
)

display(file_counts_df)

## 12. Prepare the Auto Loader landing folder

This step clears the landing directory and copies only the initial-schema files. The evolved, renamed, and malformed groups remain in staging for later notebooks.

In [0]:
dbutils.fs.rm(landing_path, recurse=True)
dbutils.fs.mkdirs(landing_path)

print("Landing folder cleared and recreated.")

In [0]:
def copy_json_files(
    source_directory: str,
    target_directory: str,
    prefix: str
) -> int:
    copied_files = 0

    for file_info in dbutils.fs.ls(source_directory):
        if not file_info.name.endswith(".json"):
            continue

        destination = (
            f"{target_directory}/{prefix}_{file_info.name}"
        )

        dbutils.fs.cp(file_info.path, destination)
        copied_files += 1

    return copied_files


copied_initial_files = copy_json_files(
    staging_initial_path,
    landing_path,
    "initial"
)

print(f"Copied initial files: {copied_initial_files}")

## 13. Final validation

In [0]:
landing_file_count = count_json_files(landing_path)

assert initial_file_count > 0, "No initial JSON files were generated."
assert landing_file_count == initial_file_count, (
    "Landing file count does not match the initial file count."
)

print("Lab 3 file generation completed successfully.")
print(f"Generated JSON files: {total_file_count}")
print(f"Initial files copied to landing: {landing_file_count}")
print("Next notebook: lab03_02_autoloader_initial_load")